# 🎮 Video Game Trivia RAG System

A Retrieval-Augmented Generation (RAG) application that answers questions about video games using a knowledge base of game information.

## 🔧 Import Libraries

In [ ]:
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from typing import List, Tuple
import json
import os
import random

## 🎲 Create Game Knowledge Base

In [ ]:
# Create a comprehensive game knowledge base
game_data = [
    {
        "title": "The Legend of Zelda: Breath of the Wild",
        "release_year": 2017,
        "developer": "Nintendo EPD",
        "publisher": "Nintendo",
        "genre": "Action-Adventure",
        "platform": "Nintendo Switch, Wii U",
        "description": "Open-world game where Link awakens from a 100-year slumber to defeat Calamity Ganon and save Hyrule.",
        "facts": [
            "First open-world Zelda game",
            "Features over 100 shrines to complete",
            "Won Game of the Year at The Game Awards 2017"
        ]
    },
    {
        "title": "Red Dead Redemption 2",
        "release_year": 2018,
        "developer": "Rockstar Games",
        "publisher": "Rockstar Games",
        "genre": "Action-Adventure, Western",
        "platform": "PlayStation 4, Xbox One, PC",
        "description": "Epic western tale following outlaw Arthur Morgan and the Van der Linde gang across America.",
        "facts": [
            "Development cost over $500 million",
            "Features over 200 animal species",
            "Sold over 50 million copies worldwide"
        ]
    },
    {
        "title": "Minecraft",
        "release_year": 2011,
        "developer": "Mojang Studios",
        "publisher": "Mojang Studios, Microsoft Studios",
        "genre": "Sandbox, Survival",
        "platform": "Multi-platform",
        "description": "Block-based game where players explore, build, and survive in procedurally generated worlds.",
        "facts": [
            "Best-selling video game of all time with over 300 million copies sold",
            "Originally created by Markus 'Notch' Persson",
            "Has an educational edition used in schools"
        ]
    },
    {
        "title": "The Witcher 3: Wild Hunt",
        "release_year": 2015,
        "developer": "CD Projekt Red",
        "publisher": "CD Projekt",
        "genre": "Action RPG",
        "platform": "PlayStation 4, Xbox One, PC, Nintendo Switch",
        "description": "Monster hunter Geralt searches for his adopted daughter in a war-torn fantasy world.",
        "facts": [
            "Based on Polish fantasy novels",
            "Contains over 100 hours of gameplay",
            "Won over 250 Game of the Year awards"
        ]
    },
    {
        "title": "Super Mario Odyssey",
        "release_year": 2017,
        "developer": "Nintendo EPD",
        "publisher": "Nintendo",
        "genre": "Platformer",
        "platform": "Nintendo Switch",
        "description": "Mario travels across various kingdoms in his hat-shaped ship to rescue Princess Peach.",
        "facts": [
            "Mario can possess enemies and objects with his hat",
            "Features a realistic New Donk City level",
            "Critically acclaimed for its creative level design"
        ]
    }
]

# Save to JSON for persistence
with open('game_knowledge_base.json', 'w') as f:
    json.dump(game_data, f, indent=2)

## 🧠 Initialize Embedding Model and Vector Database

In [ ]:
# Initialize the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB client
chroma_client = chromadb.Client()

# Create or get collection
try:
    collection = chroma_client.create_collection(name="game_trivia")
except:
    collection = chroma_client.get_collection(name="game_trivia")

# Prepare and add documents to the vector store
documents = []
metadata = []
ids = []

for idx, game in enumerate(game_data):
    # Create comprehensive text for each game
    game_text = f"""
    Title: {game['title']}
    Release Year: {game['release_year']}
    Developer: {game['developer']}
    Publisher: {game['publisher']}
    Genre: {game['genre']}
    Platform: {game['platform']}
    Description: {game['description']}
    Interesting Facts: {' '.join(game['facts'])}
    """
    
    documents.append(game_text)
    metadata.append({"title": game['title'], "genre": game['genre']})
    ids.append(f"game_{idx}")

# Add to collection
collection.add(
    documents=documents,
    metadatas=metadata,
    ids=ids
)

print(f"Added {len(documents)} games to the vector database")

## 🔍 RAG Query Function

In [ ]:
def retrieve_relevant_games(query: str, n_results: int = 3) -> Tuple[List[str], List[float]]:
    """
    Retrieve relevant games based on the query
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    return results['documents'][0], results['distances'][0]

def generate_response(query: str, retrieved_docs: List[str]) -> str:
    """
    Generate a response using retrieved documents
    Note: This is a simplified version without an LLM
    """
    response = f"🎮 Based on your question about '{query}', I found:\n\n"
    
    for i, doc in enumerate(retrieved_docs, 1):
        # Parse the document to extract key information
        lines = doc.strip().split('\n')
        title = lines[1].replace('Title:', '').strip()
        
        response += f"📌 **{title}**\n"
        
        # Add relevant facts based on query keywords
        if 'release' in query.lower() or 'year' in query.lower():
            year_line = [l for l in lines if 'Release Year' in l][0]
            response += f"   {year_line.strip()}\n"
        
        if 'developer' in query.lower() or 'who made' in query.lower():
            dev_line = [l for l in lines if 'Developer' in l][0]
            response += f"   {dev_line.strip()}\n"
        
        if 'genre' in query.lower() or 'type of' in query.lower():
            genre_line = [l for l in lines if 'Genre' in l][0]
            response += f"   {genre_line.strip()}\n"
        
        # Always include description for context
        desc_line = [l for l in lines if 'Description' in l][0]
        response += f"   {desc_line.strip()}\n"
        
        # Add facts if query seems to ask for interesting information
        if 'fact' in query.lower() or 'interesting' in query.lower():
            facts = [l for l in lines if 'Interesting Facts' in l][0]
            response += f"   {facts.strip()}\n"
        
        response += "\n"
    
    return response

def rag_query(query: str) -> str:
    """
    Main RAG query function that retrieves and generates responses
    """
    if not query or query.strip() == "":
        return "Please enter a question about video games!"
    
    # Retrieve relevant documents
    retrieved_docs, distances = retrieve_relevant_games(query)
    
    # Generate response
    response = generate_response(query, retrieved_docs)
    
    return response

## 🎯 Game-Specific Functions

In [ ]:
def get_game_comparison(game1: str, game2: str) -> str:
    """
    Compare two games using RAG
    """
    query = f"Compare {game1} and {game2}"
    retrieved_docs, _ = retrieve_relevant_games(query, n_results=4)
    
    # Extract information for both games
    game1_info = ""
    game2_info = ""
    
    for doc in retrieved_docs:
        lines = doc.strip().split('\n')
        title = lines[1].replace('Title:', '').strip()
        
        if game1.lower() in title.lower() or title.lower() in game1.lower():
            game1_info = doc
        elif game2.lower() in title.lower() or title.lower() in game2.lower():
            game2_info = doc
    
    if not game1_info or not game2_info:
        return "Could not find information for one or both games."
    
    # Generate comparison
    comparison = "### 🎮 Game Comparison\n\n"
    
    lines1 = game1_info.strip().split('\n')
    lines2 = game2_info.strip().split('\n')
    
    comparison += f"**{lines1[1].replace('Title:', '').strip()}** vs **{lines2[1].replace('Title:', '').strip()}**\n\n"
    
    # Compare key features
    features = ['Release Year', 'Developer', 'Genre']
    
    for feature in features:
        f1 = [l for l in lines1 if feature in l][0].replace(feature + ':', '').strip()
        f2 = [l for l in lines2 if feature in l][0].replace(feature + ':', '').strip()
        comparison += f"• **{feature}**: {f1} | {f2}\n"
    
    return comparison

def get_random_game_fact() -> str:
    """
    Get a random interesting fact about a game
    """
    
    game = random.choice(game_data)
    fact = random.choice(game['facts'])
    
    return f"**{game['title']}**: {fact}"

## 🎨 Create Gradio Interface

In [ ]:
# Create the Gradio interface with multiple tabs
with gr.Blocks(title="🎮 Video Game RAG Trivia", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎮 Video Game RAG Trivia System")
    gr.Markdown("Ask questions about video games and get AI-powered responses based on our knowledge base!")
    
    with gr.Tab("🔍 Ask Questions"):
        with gr.Row():
            with gr.Column(scale=3):
                query_input = gr.Textbox(
                    label="Your Question",
                    placeholder="e.g., What's the best-selling game of all time? or Tell me about The Witcher 3",
                    lines=2
                )
                submit_btn = gr.Button("Ask RAG", variant="primary")
            
            with gr.Column(scale=2):
                example_questions = gr.Examples(
                    examples=[
                        "Who developed The Legend of Zelda: Breath of the Wild?",
                        "What are some interesting facts about Minecraft?",
                        "When was Red Dead Redemption 2 released?",
                        "Tell me about Super Mario Odyssey",
                        "What genre is The Witcher 3?"
                    ],
                    inputs=query_input
                )
        
        with gr.Row():
            output = gr.Markdown(label="RAG Response")
        
        submit_btn.click(
            fn=rag_query,
            inputs=query_input,
            outputs=output
        )
    
    with gr.Tab("⚔️ Compare Games"):
        with gr.Row():
            game1_input = gr.Textbox(label="First Game", placeholder="e.g., Minecraft")
            game2_input = gr.Textbox(label="Second Game", placeholder="e.g., The Witcher 3")
        
        compare_btn = gr.Button("Compare Games", variant="primary")
        comparison_output = gr.Markdown(label="Comparison")
        
        compare_btn.click(
            fn=get_game_comparison,
            inputs=[game1_input, game2_input],
            outputs=comparison_output
        )
    
    with gr.Tab("🎲 Random Facts"):
        gr.Markdown("Click the button to get a random interesting fact about a video game!")
        fact_btn = gr.Button("Get Random Fact", variant="primary")
        fact_output = gr.Markdown(label="Random Fact")
        
        fact_btn.click(
            fn=get_random_game_fact,
            inputs=[],
            outputs=fact_output
        )
    
    with gr.Tab("📚 Knowledge Base"):
        gr.Markdown("### Games in our database:")
        
        # Display games in a nice format
        for game in game_data:
            with gr.Row():
                with gr.Column():
                    gr.Markdown(f"**{game['title']}** ({game['release_year']})")
                    gr.Markdown(f"*{game['developer']}* | {game['genre']}")
                    gr.Markdown(f"_{game['description']}_")
            gr.Markdown("---")
    
    gr.Markdown("---")
    gr.Markdown("### ℹ️ How it works:")
    gr.Markdown("""
    1. Your question is converted into a vector embedding
    2. We retrieve the most relevant game information from our vector database
    3. The retrieved information is formatted into a comprehensive response
    4. All answers are based on our curated knowledge base of popular video games
    """)

# Launch the app
demo.launch(inbrowser=True)

## 🧪 Test the RAG System

In [ ]:
# Quick test of the RAG system
test_queries = [
    "Who developed Minecraft?",
    "What's interesting about The Witcher 3?",
    "Tell me about Zelda games"
]

print("=== Testing RAG System ===\n")
for query in test_queries:
    print(f"Q: {query}")
    print(f"A: {rag_query(query)}")
    print("-" * 50)